In [ ]:
# Clonare il repository YOLOv10
!git clone https://github.com/THU-MIG/yolov10.git

# Navigare nella directory del repository
%cd yolov10

# Installare il pacchetto
!pip install .

Cloning into 'yolov10'...
remote: Enumerating objects: 20329, done.
remote: Counting objects: 100% (2443/2443), done.
remote: Compressing objects: 100% (245/245), done.
remote: Total 20329 (delta 2315), reused 2198 (delta 2198), pack-reused 17886 (from 1)
Receiving objects: 100% (20329/20329), 11.14 MiB | 5.47 MiB/s, done.
Resolving deltas: 100% (14335/14335), done.
/content/yolov10
Processing /content/yolov10
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.1.34-py3-none-any.whl size=731411 sha256=b6ef3dd2c6123f961b0246f9dd6b153146639fc05ed8bee4ae47f0f89efa84a3
  Stored in directory: /tmp/pip-ephem-wheel-cache-9i2k4g3q/wheels/51/93/e8/22d2e815ced343915c15d86b2a00d95eb0a997d012527fbea7
Successfully built ultralytics


In [ ]:
from ultralytics import YOLOv10

In [ ]:
# Decomprimi la cartella yolo_cam.zip
!unzip -q /content/yolo_cam.zip

In [ ]:
!pip install ttach

In [ ]:
import os
import time
import argparse
import cv2
# from ultralytics import YOLOv10
import numpy as np
from PIL import Image
from yolo_cam.eigen_cam import EigenCAM
from yolo_cam.utils.image import show_cam_on_image


def get_unique_results_dir(base_dir="results", name_dir="run"):
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)

    n = 0
    while True:
        results_dir = os.path.join(base_dir, f"{name_dir}_{n}")
        if not os.path.exists(results_dir):
            os.makedirs(results_dir, exist_ok=True)
            return results_dir
        n += 1


def infer_image(weights_path, image_path, base_dir):
    # Load the model
    model = YOLOv10(weights_path)
    save_name_img = weights_path.split("/")[-1].split(".pt")[0]
    # Create unique results directory
    results_dir = get_unique_results_dir(base_dir=base_dir, name_dir=save_name_img)

    # Load the image
    image = Image.open(image_path)
    image_np = np.array(image)
    image_cv = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    image_rgb = np.float32(image_np) / 255
    norm_img_path = os.path.join(results_dir, f"{save_name_img}_base.jpg")
    cv2.imwrite(norm_img_path, image_cv)

    # Measure inference time
    start_time = time.time()
    results = model(image_path)
    end_time = time.time()
    inference_time = end_time - start_time

    with open(os.path.join(results_dir, f"{save_name_img}_results.txt"), "w") as file:
        # Write attributes of the results object to the file
        for result in results:
            file.write(f"boxes: {str(result.boxes)}\n")
            file.write(f"keypoints: {str(result.keypoints)}\n")
            file.write(f"masks: {str(result.masks)}\n")
            file.write(f"names: {str(result.names)}\n")
            file.write(f"orig_shape: {str(result.orig_shape)}\n")
            file.write(f"speed: {str(result.speed)}\n")

    # Save inference results with bounding boxes
    for i, result in enumerate(results):
        result_img = result.plot()
        result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        result_img = Image.fromarray(result_img)
        result_path = os.path.join(results_dir, f"{save_name_img}_inference.jpg")
        result_img.save(result_path)

    # Generate the CAM for the full image
    target_layer = model.model.model[-2]
    cam = EigenCAM(model, target_layers=[target_layer], task='od')
    grayscale_cam = cam(image_rgb)[0, :, :]
    cam_image = show_cam_on_image(image_rgb, grayscale_cam, use_rgb=True)
    cam_image_bgr = cv2.cvtColor(cam_image, cv2.COLOR_RGB2BGR)

    # Save the EigenCAM result for the full image
    eigen_cam_path = os.path.join(results_dir, f"{save_name_img}_eigen_cam_full.jpg")
    cv2.imwrite(eigen_cam_path, cam_image_bgr)

    # Save inference details to a text file
    with open(os.path.join(results_dir, f"{save_name_img}_time.txt"), "w") as f:
        f.write(f"end   Time: {end_time:.4f} seconds  - \n")
        f.write(f"start Time: {start_time:.4f} seconds  = \n")
        f.write(f"--------------------------------------------\n")
        f.write(f"Inference Time: {inference_time:.4f} seconds\n")

    print(f"Results saved in: {results_dir}")
    print(f"Inference Time: {inference_time:.4f} seconds\n")


In [ ]:
# Specifica il percorso al tuo modello e immagine
weights_path = '/yolov10s_200e_64b_SGD_best.pt'
image_path = '/nave_2.jpg'
results_dir = '/content/results'  # Directory dove salvare i risultati

# Chiama la funzione infer_image con i percorsi specificati
infer_image(weights_path, image_path, results_dir)


image 1/1 /nave_2.jpg: 640x640 2 ships, 25.7ms
Speed: 7.0ms preprocess, 25.7ms inference, 5.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 20.1ms
Speed: 7.4ms preprocess, 20.1ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved in: /content/results/yolov10s_200e_64b_SGD_best_1
Inference Time: 0.4183 seconds

